<a href="https://colab.research.google.com/github/Mr3242-Scripter/Roblox-useful-scripts-and-instructions/blob/main/Avanced%20badge%20deletion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Roblox Batch Badge Deletion Script

This script is designed to delete *all* (except blacklisted badges from a specific universe id you can enter) your Roblox badges. **Before running, please follow the crucial step below to insert your Roblox security cookie.**

⚠️ **Important:** Roblox may still block deletion depending on current API permissions. This is the correct method, but not every account/game supports it. This script will attempt to delete all badges found for your account.

### &#128272; **CRUCIAL STEP: Add your Roblox Security Cookie (KEEP PRIVATE!)**

**You MUST replace `"YOUR_COOKIE_HERE"` in the code cell below with your actual `.ROBLOSECURITY` cookie value.** This cookie is necessary for authentication. Never share this cookie with anyone as it grants access to your Roblox account.

#### **Option 1: Desktop (Chrome, Edge, Firefox)**
1. Go to `roblox.com` and log in.
2. Open Developer Tools (`F12` or `Ctrl+Shift+I`).
3. Go to the **Application** (Chrome/Edge) or **Storage** (Firefox) tab.
4. Under **Cookies**, select `https://www.roblox.com`.
5. Find the cookie named `.ROBLOSECURITY` and copy the long string in the 'Value' field.

#### **Option 2: iOS (iPhone/iPad)**
1. Download the **Orion Browser** from the App Store.
2. Install the **Cookie-editor** extension within Orion.
3. Log into `roblox.com` in Orion.
4. Open the Cookie-editor extension, find `.ROBLOSECURITY`, and copy its value.

In [ ]:
import requests
import time

# --- CONFIGURATION ---
ROBLOSECURITY = "TOKEN"

# Game / Universe IDs to KEEP
BADGE_DELETOR_BLACKLIST = {
    2380077519,   # Slap Battles
    5554050692,   # Play as Baldi
}

universe_cache = {
    6403373529: 2380077519,   # Slap Battles
    16068228913: 5554050692,  # Play as Baldi
}

kept_places = {
    6403373529,    # Slap Battles
    16068228913,   # Play as Baldi
}

headers = {
    "User-Agent": "Mozilla/5.0",
    "Cookie": f".ROBLOSECURITY={ROBLOSECURITY}"
}

def get_csrf():
    try:
        r = requests.post(
            "https://auth.roblox.com/v2/logout",
            headers=headers
        )
        return r.headers.get("x-csrf-token")
    except Exception:
        return None

def get_universe_id(place_id):
    if place_id in universe_cache:
        return universe_cache[place_id]

    try:
        r = requests.get(
            f"https://apis.roblox.com/universes/v1/places/{place_id}/universe",
            timeout=10
        )
        if r.status_code == 429:
            print(f"[RATE LIMIT] Universe lookup for Place {place_id} - waiting 5s...")
            time.sleep(5)
            return None
        if r.status_code != 200:
            print(f"[GAME LOOKUP ERROR] Place {place_id} - Status {r.status_code}")
            return None

        data = r.json()
        universe_id = data.get("universeId")
        if universe_id is None:
            return None

        universe_id = int(universe_id)
        universe_cache[place_id] = universe_id
        return universe_id
    except Exception as e:
        print(f"[GAME LOOKUP ERROR] Place {place_id} - {e}")
        return None

def run_deletion():
    try:
        auth_res = requests.get(
            "https://users.roblox.com/v1/users/authenticated",
            headers=headers
        )
        auth_res.raise_for_status()
        user_id = auth_res.json()["id"]
        print(f"Authenticated as: {user_id}")
    except Exception as e:
        print(f"Auth failed: {e}. Check cookie.")
        return

    cursor = None
    while True:
        print("\nFetching badges...")
        deleted_any = False

        url = (
            f"https://badges.roblox.com/v1/users/{user_id}/badges"
            f"?limit=100"
        )
        if cursor:
            url += f"&cursor={cursor}"

        badges_res = requests.get(url, headers=headers)
        data = badges_res.json()
        badges = data.get("data", [])

        if not badges:
            print("No more badges to process!")
            break

        print(f"Processing {len(badges)} badges...")

        for b in badges:
            badge_id = b["id"]

            # Get the Place ID that awarded the badge
            awarder = b.get("awarder", {})
            if awarder.get("type") != "Place":
                print(f"[SKIPPED] Badge {badge_id} - Unknown awarder type")
                continue

            place_id = awarder.get("id")
            if not place_id:
                print(f"[SKIPPED] Badge {badge_id} - No Place ID found")
                continue

            if place_id in kept_places:
                continue   # silent skip for already-known kept games

            # Convert Place ID -> Universe/Game ID
            universe_id = get_universe_id(place_id)
            if universe_id is None:
                print(f"[SKIPPED] Badge {badge_id} - Could not determine game ID")
                continue

            # BLACKLIST CHECK
            if universe_id in BADGE_DELETOR_BLACKLIST:
                print(f"[KEEP] Badge {badge_id} - Game ID: {universe_id}")
                kept_places.add(place_id)
                continue

            # DELETE BADGE
            print(f"[DELETE] Badge {badge_id} - Game ID: {universe_id}")
            success = False
            while not success:
                token = get_csrf()
                current_headers = headers.copy()
                current_headers["X-CSRF-TOKEN"] = token

                res = requests.delete(
                    f"https://badges.roblox.com/v1/user/badges/{badge_id}",
                    headers=current_headers
                )

                if res.status_code == 200:
                    print(f"[SUCCESS] Deleted: {badge_id}")
                    success = True
                    deleted_any = True
                elif res.status_code == 429:
                    print(f"[RATE LIMIT] Waiting 5s for {badge_id}...")
                    time.sleep(5)
                else:
                    print(f"[RETRYING] {badge_id} - Status {res.status_code}")
                    time.sleep(1)

            time.sleep(0.5)  # small gap to reduce 429s

        # Move to next page
        cursor = data.get("nextPageCursor")
        if not cursor:
            print("Reached the last page!")
            break

run_deletion()